<a href="https://colab.research.google.com/github/yumna-09/FlyRank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yumna-09/FlyRank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Finding 1 — Health Score feature-importance chart (paper p.27)**
Random Forest ranks impressions, avg_position, CTR, scroll_depth as top predictors of the Health Score.
Methodology question: these 4 features are the exact inputs the Health Score formula (p.5) is built from —
30/30/20/20 weights. So does this chart reveal what makes content good, or just confirm the model can
reconstruct the formula it was given? Label source and feature source overlap here — that's circular,
not causal evidence.

**Finding 2 — 71% accuracy, 80/20 split (paper p.29)**
Paper reports 71/100 correct on an 80/20 holdout, doesn't say if split was row-random or client-grouped.
Methodology question: were pages from the same brand present in both train and test? If row-random,
71% may reflect the model memorizing brand-specific patterns rather than a generalizable signal.
Base rate (guess-only accuracy) also isn't reported — without it, 71% can't be judged as meaningfully
better than chance.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "FlyRank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "https://github.com/yumna-09/FlyRank-ML-Internship.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

features = ["impressions_90d", "ctr", "avg_position", "engagement_rate", "content_age_days"]
X = df[features].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scores = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)
best_k = max(scores, key=scores.get)
print("best_k rebuilt:", best_k)

groups = df.loc[X.index, "client_id"]

rng = np.random.RandomState(42)
unique_clients = groups.unique()
rng.shuffle(unique_clients)
split_point = int(len(unique_clients) * 0.8)
train_clients = set(unique_clients[:split_point])

train_mask = groups.isin(train_clients).values
test_mask = ~train_mask

X_train = X_scaled[train_mask]
X_test = X_scaled[test_mask]

km_all = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(X_scaled)
sil_all = silhouette_score(X_scaled, km_all.labels_)

km_train = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(X_train)
test_labels = km_train.predict(X_test)
sil_test = silhouette_score(X_test, test_labels)

print("All-data silhouette (before, leaky):", round(sil_all, 4))
print("Held-out-client silhouette (after, honest):", round(sil_test, 4))
print("Train clients:", len(train_clients), "| Test clients:", len(unique_clients) - len(train_clients))

Working dir: /content/FlyRank-ML-Internship/FlyRank-ML-Internship/FlyRank-ML-Internship
best_k rebuilt: 7
All-data silhouette (before, leaky): 0.3763
Held-out-client silhouette (after, honest): 0.3046
Train clients: 25 | Test clients: 7


My Week-5 lane used unsupervised KMeans clustering (no label), fit on all 30k rows at once —
that's leaky: silhouette score was computed on the same data the centroids were tuned on.

Honest check: fit centroids on 80% of client_ids (25 clients) only, then score on the held-out
20% (7 clients) never seen during fit. This tests whether clusters generalize to unseen clients,
not just memorize the training set's shape.

**Results:**
- All-data silhouette (before, leaky): 0.3763
- Held-out-client silhouette (after, honest): 0.3046

The score drops from 0.3763 to 0.3046 under the honest split — expected, not a bug. The
all-data number was inflated because the same clients' pages shaped both the cluster centroids
and the score used to judge them. 0.3046 on 7 completely unseen clients is the more trustworthy
estimate of how well these clusters generalize to new clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
scores_train_only = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_train)
    scores_train_only[k] = silhouette_score(X_train, km.labels_)

print("Silhouette by k (train clients only):", {k: round(v, 4) for k, v in scores_train_only.items()})
best_k_train_only = max(scores_train_only, key=scores_train_only.get)
print("Best k on train-only:", best_k_train_only, "| Best k used in Week 5 (full data):", best_k)

print("\nFeatures used:", features)
print("client_id / content_id used as feature?", any(c in features for c in ["client_id", "content_id"]))

Silhouette by k (train clients only): {2: np.float64(0.3245), 3: np.float64(0.3615), 4: np.float64(0.3795), 5: np.float64(0.396), 6: np.float64(0.4), 7: np.float64(0.4095)}
Best k on train-only: 7 | Best k used in Week 5 (full data): 7

Features used: ['impressions_90d', 'ctr', 'avg_position', 'engagement_rate', 'content_age_days']
client_id / content_id used as feature? False


Leakage risk here isn't target leakage (no target used) — it's k-selection leakage: best_k=7
was chosen using silhouette scored on the full dataset, then "validated" on that same full
dataset. Auditing that by re-running k-selection using train-clients-only data.

**Result:** best_k on train-clients-only data = 7, same as best_k chosen on the full dataset (7).
No k-selection leakage found here — the choice of k=7 is stable whether or not the held-out
clients are included in the search. (Note: silhouette values themselves are still lower on the
smaller train-only set at every k, which is expected — smaller sample, more variance — but the
*ranking* of k values, and the winner, didn't change.)

Second check: confirmed `client_id` and `content_id` are not in the feature list used for
clustering (`features = ['impressions_90d', 'ctr', 'avg_position', 'engagement_rate',
'content_age_days']`) — no identifier leaked in as a signal.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import silhouette_samples

sample_scores_test = silhouette_samples(X_test, test_labels)
pct_negative_test = (sample_scores_test < 0).mean() * 100

print("N held-out test clients:", len(unique_clients) - len(train_clients))
print("N held-out test pages:", test_mask.sum())
print("%% of held-out points with negative silhouette (poorly matched):", round(pct_negative_test, 1))

df_test = df.loc[X.index[test_mask]].copy()
df_test["cluster"] = test_labels
df_test["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int).loc[df_test.index]

print("\nDecline rate by cluster (held-out clients only, label not used in clustering):")
print(df_test.groupby("cluster")["is_declining_label"].mean().round(3))

N held-out test clients: 7
N held-out test pages: 7611
%% of held-out points with negative silhouette (poorly matched): 4.7

Decline rate by cluster (held-out clients only, label not used in clustering):
cluster
0    0.398
1    0.456
2    0.619
3    0.498
4    0.263
5    0.475
6    0.146
Name: is_declining_label, dtype: float64


**Old (Week 5) claim:** "KMeans clearly finds more natural, well-separated groups than the
rule-based buckets did."

**Rewritten, safe language:**
Observed: on 7 held-out clients (7,611 pages) not seen during clustering, KMeans (k=7) still
separates content into distinct behavioral groups, though the held-out silhouette score
(0.3046) is lower than the all-data estimate (0.3763) — see Section 2. Directional — separation
quality varies by cluster: 4.7% of held-out points were poorly matched (negative silhouette),
so not every page fits its assigned cluster cleanly.

As a descriptive check only (label not used in clustering), decline rate varies a lot by
cluster on held-out clients — from 14.6% (cluster 6) to 61.9% (cluster 2). This is interesting
but not evidence the clustering causes or predicts decline; it's an observed association on
7 clients, too small a sample to generalize confidently.

Decision-support only: clusters group pages with similar behavior for review prioritization —
they do not predict future traffic or performance changes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.